In [1]:
import os
import pandas as pd
import requests

In [4]:
# 1. Configuration parameters
# Lat/Lon for center of Narail District, Bangladesh
LATITUDE = 23.1667
LONGITUDE = 89.5000
# Target Time Window: 2-Year Historical Frame (Jan 2023 - Dec 2024)
START_DATE = "20230101"
END_DATE = "20241231"
# Parameters mapping to system attributes:
# ALLSKY_SFC_SW_DWN = Global Horizontal Irradiance (GHI) - Core input for Solar PV prediction
# CLRSKY_SFC_SW_DWN = Clear Sky Solar Irradiance - Essential for computing cloud attenuation
# T2M               = Ambient Temperature at 2m - Governing factor for PV thermal efficiency and digester heating
# RH2M              = Relative Humidity at 2m
# WS10M             = Wind Speed at 10m - Controls ambient convective cooling
PARAMS = ["ALLSKY_SFC_SW_DWN", "CLRSKY_SFC_SW_DWN", "T2M", "RH2M", "WS10M"]

In [ ]:
def fetch_nasa_power_data(lat, lon, start, end, parameters):
    url = (
        f"https://power.larc.nasa.gov/api/temporal/daily/point?"
        f"parameters={','.join(parameters)}&"
        f"community=RE&"  # Renewable Energy community profile
        f"longitude={lon}&"
        f"latitude={lat}&"
        f"start={start}&"
        f"end={end}&"
        f"format=JSON"
    )

    print(f"Connecting to NASA API for Coordinates: Lat {lat}, Lon {lon}...")
    response = requests.get(url, timeout=30)

    if response.status_code == 200:
        json_data = response.json()
        # Parse parameter data dictionary
        raw_records = json_data["properties"]["parameter"]

        # Convert nested dictionaries into standard DataFrame
        df = pd.DataFrame(raw_records)
        df.index = pd.to_datetime(df.index, format="%Y%m%d")
        df.index.name = "Date"

        # Relabel parameters into highly descriptive engineering headers
        df = df.rename(
            columns={
                "ALLSKY_SFC_SW_DWN": "GHI_Satellite",
                "CLRSKY_SFC_SW_DWN": "ClearSky_GHI",
                "T2M": "Ambient_Temp",
                "RH2M": "Humidity",
                "WS10M": "Wind_Speed",
            }
        )
        return df
    else:
        raise Exception(
            f"API Connection Failed. Status: {response.status_code}, Msg: {response.text}"
        )


In [6]:
try:
    master_weather_df = fetch_nasa_power_data(
        LATITUDE, LONGITUDE, START_DATE, END_DATE, PARAMS
    )
    master_weather_df.to_csv("data/downloaded_weather_base.csv")
    print("\nInitialization Complete!")
    print(f"Successfully saved {len(master_weather_df)} records.")
    print(master_weather_df.head())
except Exception as e:
    print(f"Execution Error: {e}")

Connecting to NASA API for Coordinates: Lat 23.1667, Lon 89.5...

Initialization Complete!
Successfully saved 731 records.
            GHI_Satellite  ClearSky_GHI  Ambient_Temp  Humidity  Wind_Speed
Date                                                                       
2023-01-01         3.0077        3.3367         17.16     76.93        2.38
2023-01-02         3.2676        3.3418         17.21     76.45        2.59
2023-01-03         1.5967        3.6482         16.71     76.99        2.79
2023-01-04         1.1642        3.5674         15.45     77.22        3.06
2023-01-05         2.7022        3.5978         15.23     75.65        3.32


In [7]:
def generate_proxy_targets(input_path, output_path):
    # Load the downloaded weather data
    df = pd.read_csv(input_path, parse_dates=["Date"], index_col="Date")

    # ----------------------------------------------------
    # 1. SOLAR ENERGY OUTPUT SIMULATION (E_solar)
    # ----------------------------------------------------
    # PV temperature coefficient (gamma) = -0.004 (typical silicon panel loss per degree above 25°C)
    gamma = 0.004
    performance_ratio = 0.75  # Typical system efficiency losses

    # Plant A: 2 kW capacity
    df["PlantA_E_solar"] = (
        2.0
        * df["GHI_Satellite"]
        * performance_ratio
        * (1 - gamma * (df["Ambient_Temp"] - 25))
    )

    # Plant B: 10 kW capacity
    df["PlantB_E_solar"] = (
        10.0
        * df["GHI_Satellite"]
        * performance_ratio
        * (1 - gamma * (df["Ambient_Temp"] - 25))
    )

    # ----------------------------------------------------
    # 2. BIOGAS ENERGY OUTPUT SIMULATION (E_biogas)
    # ----------------------------------------------------
    # Biogas production lags behind temperature due to slurry thermal inertia.
    # We will simulate a 14-day rolling window to capture the hydrolysis delay.
    smoothed_temp = df["Ambient_Temp"].rolling(window=14, min_periods=1).mean()

    # Define seasonal baseline feeding fluctuations (Monsoon seasonality effect)
    # Month 6, 7, 8 (June-August) represent high monsoon substrate enrichment
    df["Month"] = df.index.month
    feed_factor = df["Month"].apply(lambda m: 1.2 if m in [6, 7, 8] else 1.0)

    # Plant A: 15 m³ digester baseline output (approx 5 to 15 kWh daily max)
    df["PlantA_E_biogas"] = (smoothed_temp * 0.35) * feed_factor + np.random.normal(
        0, 0.5, len(df)
    )

    # Plant B: 60 m³ digester baseline output (approx 20 to 60 kWh daily max)
    df["PlantB_E_biogas"] = (smoothed_temp * 1.4) * feed_factor + np.random.normal(
        0, 1.5, len(df)
    )

    # Clip any negative outputs introduced by the random noise to 0
    target_cols = [
        "PlantA_E_solar",
        "PlantB_E_solar",
        "PlantA_E_biogas",
        "PlantB_E_biogas",
    ]
    df[target_cols] = df[target_cols].clip(lower=0)

    # Drop intermediate helping columns
    df = df.drop(columns=["Month"])

    # Save the prepared dataset
    df.to_csv(output_path)
    print(f"Target simulation complete! Dataset saved to: {output_path}")
    print(df[target_cols].head())


In [8]:
def apply_bias_correction(input_path, output_path):
    df = pd.read_csv(input_path, parse_dates=["Date"], index_col="Date")

    # Extract month to apply seasonal bias correction factors
    # Satellites routinely overestimate monsoon GHI due to complex cloud dynamics in Bangladesh
    df["Month"] = df.index.month

    # Define empirical correction factors based on regional ground-station benchmarks
    # (e.g., scaling down raw satellite GHI during heavy monsoon/dust periods)
    def get_correction_factor(month):
        if month in [6, 7, 8]:  # Monsoon months
            return 0.88  # 12% overestimation correction
        elif month in [12, 1, 2]:  # Winter fog/haze months
            return 0.93  # 7% correction for fog attenuation
        else:
            return 0.98  # Standard clear-sky minor adjustment

    df["Correction_Factor"] = df["Month"].apply(get_correction_factor)

    # Create the true Bias-Corrected GHI feature
    df["GHI_Bias_Corrected"] = df["GHI_Satellite"] * df["Correction_Factor"]

    # Recalculate your targets using the corrected, realistic weather input
    performance_ratio = 0.75
    gamma = 0.004

    df["PlantA_E_solar"] = (
        2.0
        * df["GHI_Bias_Corrected"]
        * performance_ratio
        * (1 - gamma * (df["Ambient_Temp"] - 25))
    )
    df["PlantB_E_solar"] = (
        10.0
        * df["GHI_Bias_Corrected"]
        * performance_ratio
        * (1 - gamma * (df["Ambient_Temp"] - 25))
    )

    # Drop helping columns and overwrite
    df = df.drop(columns=["Month", "Correction_Factor"])
    df.to_csv(output_path)

    print("Bias correction successfully applied to the database features!")
    print(df[["GHI_Satellite", "GHI_Bias_Corrected", "PlantA_E_solar"]].head())


In [9]:
apply_bias_correction("data/processed_dataset.csv", "data/processed_dataset.csv")

Bias correction successfully applied to the database features!
            GHI_Satellite  GHI_Bias_Corrected  PlantA_E_solar
Date                                                         
2023-01-01         3.0077            2.797161        4.327320
2023-01-02         3.2676            3.038868        4.700339
2023-01-03         1.5967            1.484931        2.301257
2023-01-04         1.1642            1.082706        1.686098
2023-01-05         2.7022            2.513046        3.916884
